# Devcice_logs to Bronze

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
device_logs = spark.read.json('/Volumes/telecom_catalog/default/landing/Telecom_data/device_logs/')

In [ ]:
device_logs.display(5)

In [ ]:
device_logs.printSchema()

In [ ]:

payload_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("region", StringType(), True),
    StructField("cpu_usage", DoubleType(), True),
    StructField("memory_usage", DoubleType(), True),
    StructField("packet_loss", IntegerType(), True),
    StructField("latency_ms", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("temp", DoubleType(), True)
])

In [ ]:

parsed_df = device_logs.withColumn(
    "payload",
    from_json(col("raw_payload"), payload_schema)
)

In [ ]:
parsed_df.printSchema()

In [ ]:

bronze_df = parsed_df.select(
    col("payload.*"),
    "ingestion_time",
    "source_timestamp",
    "partition_date",
    "partition",
    "offset",
    "topic"
)

In [ ]:
(
    bronze_df.write.format("delta")
    .mode("append")
    .save("/Volumes/telecom_catalog/default/bronze/Device_logs")

)